In [3]:
import os
import re

def rename_lbrm_files():
    base_dir = '../metadata/LBRM_daymet/'
    if not os.path.isdir(base_dir):
        print(f"Error: Directory not found: {base_dir}")
        return

    prefix_map = {
        "ER": "eri",
        "GEO": "geo",
        "HU": "hur",
        "MIC": "mic",
        "ON": "on",
        "STC": "stc",
        "SUP": "sup"
    }

    renamed_count = 0
    skipped_count = 0

    print(f"Scanning directory: {os.path.abspath(base_dir)}")

    for filename in os.listdir(base_dir):
        if filename.endswith("_daymet.csv"):
            old_filepath = os.path.join(base_dir, filename)
            
            # Remove suffix
            name_part = filename.replace("_daymet.csv", "")
            
            # Use regex to find the prefix and number
            # This regex looks for known prefixes followed by one or more digits
            match = re.match(r"([A-Z]+)(\d+)", name_part)
            
            if match:
                original_prefix = match.group(1)
                number_str = match.group(2)
                
                if original_prefix in prefix_map:
                    new_prefix = prefix_map[original_prefix]
                    
                    # Format number: add leading zero if single digit
                    if len(number_str) == 1:
                        new_number_str = "0" + number_str
                    else:
                        new_number_str = number_str
                    
                    new_filename_base = new_prefix + new_number_str
                    new_filename = new_filename_base + ".csv"
                    new_filepath = os.path.join(base_dir, new_filename)
                    
                    if old_filepath != new_filepath:
                        try:
                            os.rename(old_filepath, new_filepath)
                            print(f"Renamed: '{filename}' -> '{new_filename}'")
                            renamed_count += 1
                        except OSError as e:
                            print(f"Error renaming '{filename}': {e}")
                            skipped_count += 1
                    else:
                        print(f"Skipped (no change): '{filename}'")
                        skipped_count += 1
                else:
                    print(f"Skipped (unknown prefix '{original_prefix}'): '{filename}'")
                    skipped_count += 1
            else:
                print(f"Skipped (pattern not matched): '{filename}'")
                skipped_count += 1
        elif filename.endswith(".csv"):
             # Catch files that might already be partially renamed or don't fit the full pattern
            print(f"Skipped (does not match '_daymet.csv' suffix pattern): '{filename}'")
            skipped_count +=1


    print(f"\nRenaming complete. {renamed_count} files renamed, {skipped_count} files skipped.")

# To run the function:
rename_lbrm_files()

Scanning directory: c:\Users\ybrot\Desktop\course\UROP\GAGEii_modeling\metadata\LBRM_daymet
Renamed: 'ER0_daymet.csv' -> 'eri00.csv'
Renamed: 'ER10_daymet.csv' -> 'eri10.csv'
Renamed: 'ER11_daymet.csv' -> 'eri11.csv'
Renamed: 'ER12_daymet.csv' -> 'eri12.csv'
Renamed: 'ER13_daymet.csv' -> 'eri13.csv'
Renamed: 'ER14_daymet.csv' -> 'eri14.csv'
Renamed: 'ER15_daymet.csv' -> 'eri15.csv'
Renamed: 'ER16_daymet.csv' -> 'eri16.csv'
Renamed: 'ER17_daymet.csv' -> 'eri17.csv'
Renamed: 'ER18_daymet.csv' -> 'eri18.csv'
Renamed: 'ER19_daymet.csv' -> 'eri19.csv'
Renamed: 'ER1_daymet.csv' -> 'eri01.csv'
Renamed: 'ER20_daymet.csv' -> 'eri20.csv'
Renamed: 'ER21_daymet.csv' -> 'eri21.csv'
Renamed: 'ER2_daymet.csv' -> 'eri02.csv'
Renamed: 'ER3_daymet.csv' -> 'eri03.csv'
Renamed: 'ER4_daymet.csv' -> 'eri04.csv'
Renamed: 'ER5_daymet.csv' -> 'eri05.csv'
Renamed: 'ER6_daymet.csv' -> 'eri06.csv'
Renamed: 'ER7_daymet.csv' -> 'eri07.csv'
Renamed: 'ER8_daymet.csv' -> 'eri08.csv'
Renamed: 'ER9_daymet.csv' -> 'eri09

In [17]:
import csv
import datetime
import os
import pandas as pd

def process_flw_to_filtered_csv(flw_filepath, output_csv_filepath, 
                                filter_start_date_str, filter_end_date_str):
    """
    Extracts daily flow data from a .flw file, filters it by a date range,
    ensures all consecutive dates within the range are present, and saves it to a CSV file.

    Args:
        flw_filepath (str): Path to the input .flw file.
        output_csv_filepath (str): Path to the output .csv file.
        filter_start_date_str (str): Start date for filtering (e.g., "1980-01-01").
        filter_end_date_str (str): End date for filtering (e.g., "2023-12-31").
    """
    flw_start_date_obj = None
    current_date_obj = None
    data_lines_started = False
    raw_extracted_data = [] 

    print(f"DEBUG: Attempting to process FLW file: {flw_filepath}")
    print(f"DEBUG: Filter Start: {filter_start_date_str}, Filter End: {filter_end_date_str}")

    try:
        filter_start_date = datetime.datetime.strptime(filter_start_date_str, "%Y-%m-%d").date()
        filter_end_date = datetime.datetime.strptime(filter_end_date_str, "%Y-%m-%d").date()
    except ValueError:
        print(f"Error: Invalid filter date format. Please use YYYY-MM-DD.")
        return

    try:
        # It's good practice to specify encoding
        with open(flw_filepath, 'r', encoding='utf-8') as f_in:
            # You can add a peek here if you still have issues:
            # print("DEBUG: First 5 lines of the file being processed:")
            # temp_lines = [f_in.readline() for _ in range(5)]
            # for i, l_peek in enumerate(temp_lines):
            #     print(f"DEBUG_LINE {i+1}: {repr(l_peek)}")
            # f_in.seek(0) # Reset file pointer to the beginning

            for line_number, line_content in enumerate(f_in, 1):
                line = line_content.strip() # Removes leading/trailing whitespace
                if not line: continue

                # After strip(), the line will not start with a space anymore
                if line.startswith("FROM "): # MODIFIED: Removed leading space from pattern
                    parts = line.split()
                    if len(parts) >= 4: # "FROM", YYYY, MM, DD
                        try:
                            year, month, day = int(parts[1]), int(parts[2]), int(parts[3])
                            flw_start_date_obj = datetime.date(year, month, day)
                            current_date_obj = flw_start_date_obj
                            print(f"DEBUG: Parsed FLW 'FROM' date: {flw_start_date_obj}")
                        except ValueError:
                            print(f"Warning: Could not parse date from 'FROM' line: '{line_content.strip()}'")
                            return
                    else:
                        print(f"Warning: 'FROM' line has unexpected format: '{line_content.strip()}'")
                        return

                elif line.startswith("Flow(cms)"): # MODIFIED: Removed leading space from pattern
                    if flw_start_date_obj is None:
                        print("Error: 'Flow(cms)' header found before 'FROM' date line. Aborting.")
                        return
                    data_lines_started = True
                    print(f"DEBUG: 'Flow(cms)' header found. Data lines starting. Initial current_date_obj: {current_date_obj}")
                    continue

                elif data_lines_started and current_date_obj:
                    parts = line.split() # Splits by whitespace by default
                    if not parts: continue
                    
                    if current_date_obj == filter_start_date:
                        print(f"DEBUG: Reached filter_start_date ({filter_start_date}) with current_date_obj from FLW.")
                    elif current_date_obj == filter_start_date - datetime.timedelta(days=1):
                        print(f"DEBUG: One day before filter_start_date. current_date_obj from FLW: {current_date_obj}")

                    try:
                        flow_value = float(parts[0])
                        
                        is_within_filter = False
                        if filter_start_date <= current_date_obj <= filter_end_date:
                            is_within_filter = True
                            raw_extracted_data.append((current_date_obj, flow_value))
                        
                        if line_number % 5000 == 0:
                             print(f"DEBUG: Processing line {line_number}, FLW Date: {current_date_obj}, In Filter: {is_within_filter}")

                        current_date_obj += datetime.timedelta(days=1)
                        
                        if current_date_obj > filter_end_date and len(raw_extracted_data) > 0 and raw_extracted_data[-1][0] == filter_end_date:
                            print(f"DEBUG: Optimization break: current_date_obj ({current_date_obj}) > filter_end_date and last raw data was for filter_end_date.")
                            break
                    except (ValueError, IndexError):
                        if line_number % 5000 == 0:
                            print(f"DEBUG: Line {line_number}: Could not parse flow data: '{line_content.strip()}'")
                        pass
    
    except FileNotFoundError:
        print(f"Error: Input file not found at {flw_filepath}")
        return
    except Exception as e:
        print(f"An unexpected error occurred while reading {flw_filepath}: {e}")
        return

    print(f"DEBUG: Finished reading FLW file. Number of items in raw_extracted_data: {len(raw_extracted_data)}")
    if raw_extracted_data:
        print(f"DEBUG: First extracted date: {raw_extracted_data[0][0]}, Last extracted date: {raw_extracted_data[-1][0]}")
    else:
        print(f"DEBUG: raw_extracted_data is empty.")

    if not raw_extracted_data:
        print(f"Warning: No data extracted from '{flw_filepath}' within the date range {filter_start_date_str} to {filter_end_date_str}.")
        all_dates_pd_range = pd.date_range(start=filter_start_date_str, end=filter_end_date_str, freq='D')
        df_empty_for_range = pd.DataFrame(index=all_dates_pd_range)
        df_empty_for_range['Flow'] = pd.NA 
        df_empty_for_range.index.name = 'Date'
        
        output_dir = os.path.dirname(output_csv_filepath)
        if output_dir:
            os.makedirs(output_dir, exist_ok=True)
        
        df_empty_for_range.reset_index(inplace=True)
        df_empty_for_range['Date'] = df_empty_for_range['Date'].dt.strftime('%Y-%m-%d')
        df_empty_for_range.to_csv(output_csv_filepath, index=False, na_rep='NaN')
        print(f"Created CSV '{output_csv_filepath}' with all dates in range and NaN flow values as no data was found in FLW for this range.")
        return

    df = pd.DataFrame(raw_extracted_data, columns=['Date', 'Flow'])
    df['Date'] = pd.to_datetime(df['Date'])
    df = df.set_index('Date')

    all_dates_in_filter_range = pd.date_range(start=filter_start_date_str, end=filter_end_date_str, freq='D')
    df_reindexed = df.reindex(all_dates_in_filter_range)
    df_reindexed.index.name = 'Date' 
    df_output = df_reindexed.reset_index() 
    df_output['Date'] = df_output['Date'].dt.strftime('%Y-%m-%d') 

    output_dir = os.path.dirname(output_csv_filepath)
    if output_dir: 
        os.makedirs(output_dir, exist_ok=True)

    try:
        df_output.to_csv(output_csv_filepath, index=False, na_rep='NaN') 
        print(f"Data successfully processed and saved to '{output_csv_filepath}'")
        print(f"Total records written: {len(df_output)}")
    except Exception as e:
        print(f"An error occurred while writing to {output_csv_filepath}: {e}")



if __name__ == "__main__":
    input_flw = '../metadata/LBRM_Flow/eri01.flw'
    output_folder = '../metadata/LBRM_discharge'
    
    # Extract base filename from input_flw to create output filename
    base_filename = os.path.basename(input_flw)
    output_filename_csv = os.path.splitext(base_filename)[0] + ".csv"
    output_csv = os.path.join(output_folder, output_filename_csv)

    # Specified date range
    start_date_filter = "1980-01-01" # User requested 1980/01/01, using standard YYYY-MM-DD
    end_date_filter = "2023-12-31"   # User requested 2023/12/31

    print(f"Processing '{input_flw}'...")
    print(f"Output will be saved to '{output_csv}'")
    print(f"Filtering for dates between {start_date_filter} and {end_date_filter}")

    # Create a dummy FLW file if the target doesn't exist, for testing purposes
    if not os.path.exists(input_flw):
        print(f"Warning: Input file '{input_flw}' not found. Creating a dummy file for testing.")
        os.makedirs(os.path.dirname(input_flw), exist_ok=True)
        dummy_flw_content = """Daily Runoff to eri from subbasin 01 in m3/sec
 eri           01  0.170000E+04    {area in km2}
 FROM 1979 12 28   YR MO DY
 TO   1980 01 05   YR MO DY
    dummy_count
 Flow(cms)  %gaged  #gages
     1.0   10.0       1        1979-12-28
     2.0   10.0       1
     3.0   10.0       1        1979-12-30
     4.0   10.0       1        1979-12-31 (Day before filter start)
     5.0   10.0       1        1980-01-01 (Filter start)
     6.0   10.0       1        1980-01-02
     invalid_line
     7.0   10.0       1        1980-01-03 (Missing 1980-01-04 in source)
     8.0   10.0       1        1980-01-05 (Day after this is outside filter if end is 1980-01-03)
"""
        with open(input_flw, 'w') as f:
            f.write(dummy_flw_content)
        # Adjust filter for dummy data to make sense
        # start_date_filter = "1980-01-01"
        # end_date_filter = "1980-01-04" # Test missing date filling

    process_flw_to_filtered_csv(input_flw, output_csv, start_date_filter, end_date_filter)

    # Optional: Verify the output CSV content (first few lines)
    if os.path.exists(output_csv):
        print(f"\n--- First 10 lines of '{output_csv}' ---")
        try:
            df_check = pd.read_csv(output_csv)
            print(df_check.head(10))
            print("...")
            print(df_check.tail(10))
        except Exception as e:
            print(f"Could not read output CSV for verification: {e}")

Processing '../metadata/LBRM_Flow/eri01.flw'...
Output will be saved to '../metadata/LBRM_discharge\eri01.csv'
Filtering for dates between 1980-01-01 and 2023-12-31
DEBUG: Attempting to process FLW file: ../metadata/LBRM_Flow/eri01.flw
DEBUG: Filter Start: 1980-01-01, Filter End: 2023-12-31
DEBUG: Parsed FLW 'FROM' date: 1930-10-01
DEBUG: 'Flow(cms)' header found. Data lines starting. Initial current_date_obj: 1930-10-01
DEBUG: Processing line 5000, FLW Date: 1944-06-02, In Filter: False
DEBUG: Processing line 10000, FLW Date: 1958-02-09, In Filter: False
DEBUG: Processing line 15000, FLW Date: 1971-10-19, In Filter: False
DEBUG: One day before filter_start_date. current_date_obj from FLW: 1979-12-31
DEBUG: Reached filter_start_date (1980-01-01) with current_date_obj from FLW.
DEBUG: Processing line 20000, FLW Date: 1985-06-27, In Filter: True
DEBUG: Processing line 25000, FLW Date: 1999-03-06, In Filter: True
DEBUG: Processing line 30000, FLW Date: 2012-11-12, In Filter: True
DEBUG: O

In [14]:

import pandas as pd

filepath = '../metadata/LBRM_Flow/eri01.flw'  # Replace with your actual file path
start_line_number = 7   # Data starts on line 7 (1-indexed), after the "Flow(cms) ..." header
lines_to_skip = start_line_number - 1 # Skip header lines, including the "Flow(cms)..." line

try:
    # Use delim_whitespace=True to handle space-separated values.
    # We'll also provide column names since the actual data lines don't have a header.
    # The "Flow(cms) ..." line is complex to use as a direct header for the data structure.
    # Adjust column names based on the actual structure of data lines.
    # The .flw files sometimes have a date at the end of the first data line of a month.
    # This makes the number of columns potentially variable if not handled carefully.
    # For simplicity, let's assume we are interested in the first few columns that are always present.
    
    # Option 1: Read and let pandas infer, then inspect
    # df = pd.read_csv(filepath, skiprows=lines_to_skip, delim_whitespace=True, header=None)
    # print("DataFrame with inferred columns (first 5 rows):")
    # print(df.head())
    # print("\nInferred column names/types:")
    # print(df.info())

    # Option 2: Specify names if you know the consistent columns
    # The data lines look like: `  <flow>   <%gaged>   <#gages>  [<date>]`
    # The date is not always present on each line.
    # read_csv might struggle with this "ragged" structure if trying to get all columns.
    # If you only care about the flow, %gaged, #gages:
    df = pd.read_csv(
        filepath, 
        skiprows=lines_to_skip, 
        delim_whitespace=True, 
        header=None,  # No header row in the data part being read
        names=['Flow_cms', 'Percent_Gaged', 'Num_Gages', 'Optional_Date_String'], # Provide names
        usecols=[0, 1, 2] # Example: only take the first three consistent columns
    )
    # If you want to try and get the date string when present, you might need more complex parsing
    # or read it and then process the 'Optional_Date_String' column.
    # For the task of getting a clean Date and Flow column, the custom parser
    # in `process_flw_to_filtered_csv` is generally more robust as it handles date generation.


    print(f"Successfully read space-separated data starting from line {start_line_number}.")
    print("First 5 rows of the DataFrame (selected columns):")
    print(df.head())

except FileNotFoundError:
    print(f"Error: File not found at {filepath}")
except pd.errors.EmptyDataError:
    print(f"Error: No data found after skipping {lines_to_skip} lines. The file might be shorter.")
except Exception as e:
    print(f"An error occurred: {e}")

Successfully read space-separated data starting from line 7.
First 5 rows of the DataFrame (selected columns):
   Flow_cms  Percent_Gaged  Num_Gages
0     2.203          43.71          2
1     2.203          43.71          2
2     2.203          43.71          2
3     2.203          43.71          2
4     2.203          43.71          2


C:\Users\ybrot\AppData\Local\Temp\ipykernel_13228\239020334.py:28: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  df = pd.read_csv(


In [20]:
import pandas as pd
from pathlib import Path
from tqdm import tqdm
import datetime
import os

def silent_process_flw_to_filtered_csv(flw_filepath_str, output_csv_filepath_str,
                                       filter_start_date_str, filter_end_date_str):
    """
    Extracts daily flow data from a .flw file, filters it by a date range,
    ensures all consecutive dates within the range are present, and saves it to a CSV file.
    This version is silent and does not print debug messages.
    """
    flw_filepath = Path(flw_filepath_str)
    output_csv_filepath = Path(output_csv_filepath_str)

    flw_start_date_obj = None
    current_date_obj = None
    data_lines_started = False
    raw_extracted_data = []

    try:
        filter_start_date = datetime.datetime.strptime(filter_start_date_str, "%Y-%m-%d").date()
        filter_end_date = datetime.datetime.strptime(filter_end_date_str, "%Y-%m-%d").date()
    except ValueError:
        # Invalid date format, cannot proceed for this file
        return

    try:
        with open(flw_filepath, 'r', encoding='utf-8') as f_in:
            for line_content in f_in:
                line = line_content.strip()
                if not line:
                    continue

                if line.startswith("FROM "):
                    parts = line.split()
                    if len(parts) >= 4:
                        try:
                            year, month, day = int(parts[1]), int(parts[2]), int(parts[3])
                            flw_start_date_obj = datetime.date(year, month, day)
                            current_date_obj = flw_start_date_obj
                        except ValueError:
                            return # Error parsing date
                    else:
                        return # Unexpected 'FROM' line format

                elif line.startswith("Flow(cms)"):
                    if flw_start_date_obj is None:
                        return # 'Flow(cms)' header found before 'FROM' date
                    data_lines_started = True
                    continue

                elif data_lines_started and current_date_obj:
                    parts = line.split()
                    if not parts:
                        continue
                    try:
                        flow_value = float(parts[0])
                        if filter_start_date <= current_date_obj <= filter_end_date:
                            raw_extracted_data.append((current_date_obj, flow_value))
                        
                        current_date_obj += datetime.timedelta(days=1)
                        
                        # Optimization: if current date is past filter end and last data point was for filter end
                        if current_date_obj > filter_end_date and raw_extracted_data and \
                           raw_extracted_data[-1][0] >= filter_end_date:
                            break
                    except (ValueError, IndexError):
                        # Silently ignore lines that cannot be parsed as flow data
                        pass
    except FileNotFoundError:
        return # Input file not found
    except Exception:
        return # Other unexpected error during file reading

    # Ensure output directory exists
    output_csv_filepath.parent.mkdir(parents=True, exist_ok=True)

    # Create a DataFrame with all dates in the filter range
    all_dates_pd_range = pd.date_range(start=filter_start_date_str, end=filter_end_date_str, freq='D')
    df_for_range = pd.DataFrame(index=all_dates_pd_range)
    df_for_range.index.name = 'Date'
    df_for_range['Flow'] = pd.NA # Initialize with NA

    if raw_extracted_data:
        df_extracted = pd.DataFrame(raw_extracted_data, columns=['Date', 'Flow'])
        df_extracted['Date'] = pd.to_datetime(df_extracted['Date'])
        df_extracted = df_extracted.set_index('Date')
        # Update the Flow column in df_for_range with extracted values
        df_for_range.update(df_extracted)

    df_output = df_for_range.reset_index()
    df_output['Date'] = df_output['Date'].dt.strftime('%Y-%m-%d')

    try:
        df_output.to_csv(output_csv_filepath, index=False, na_rep='NaN')
    except Exception:
        # Silently ignore errors during CSV writing for this file
        pass

def main_processing_pipeline():
    # --- Step 1: Convert .flw files to .csv in LBRM_discharge ---
    flow_dir = Path("../metadata/LBRM_Flow")
    discharge_dir = Path("../metadata/LBRM_discharge")
    discharge_dir.mkdir(parents=True, exist_ok=True)

    flw_files = list(flow_dir.glob("*.flw"))

    start_date_filter = "1980-01-01"
    end_date_filter = "2023-12-31"

    for flw_file_path in tqdm(flw_files, desc="Step 1: Converting FLW to CSV", unit="file"):
        basin_name = flw_file_path.stem
        output_csv_path = discharge_dir / f"{basin_name}.csv"
        silent_process_flw_to_filtered_csv(str(flw_file_path), str(output_csv_path),
                                           start_date_filter, end_date_filter)

    # --- Step 2: Merge converted .csv files with Daymet data ---
    daymet_dir = Path("../metadata/LBRM_daymet")

    converted_csv_files = list(discharge_dir.glob("*.csv"))

    for discharge_csv_path in tqdm(converted_csv_files, desc="Step 2: Merging CSV with Daymet", unit="file"):
        basin_name = discharge_csv_path.stem
        daymet_file_path = daymet_dir / f"{basin_name}.csv"

        if not daymet_file_path.exists():
            continue # Skip if corresponding Daymet file doesn't exist

        try:
            discharge_df = pd.read_csv(discharge_csv_path)
            if 'Date' not in discharge_df.columns or 'Flow' not in discharge_df.columns:
                continue # Skip if required columns are missing in discharge file

            daymet_df = pd.read_csv(daymet_file_path)
            if 'time' not in daymet_df.columns:
                continue # Skip if 'time' column is missing in Daymet file

            # Prepare for merge
            # 'Date' in discharge_df is already YYYY-MM-DD string from conversion step
            discharge_df['merge_date_key'] = discharge_df['Date']
            daymet_df['merge_date_key'] = pd.to_datetime(daymet_df['time']).dt.strftime('%Y-%m-%d')

            # Merge. 'Date' column from discharge_df will be used.
            merged_df = pd.merge(
                daymet_df,
                discharge_df[['merge_date_key', 'Date', 'Flow']],
                on='merge_date_key',
                how='left'
            )

            # Post-merge processing
            merged_df.rename(columns={'Flow': 'discharge'}, inplace=True)

            daymet_original_cols = [col for col in daymet_df.columns if col not in ['time', 'merge_date_key']]
            
            # Define final column order, ensuring 'Date' (from discharge_df) is first
            final_columns = ['Date'] + daymet_original_cols + ['discharge']
            
            # Ensure all specified final columns exist in merged_df, adding them with NA if missing
            # This is a safeguard, typically merge should handle this.
            for col in final_columns:
                if col not in merged_df.columns:
                    merged_df[col] = pd.NA
            
            output_df = merged_df[final_columns]

            # Save, overwriting the Daymet file
            output_df.to_csv(daymet_file_path, index=False, na_rep='NaN')

        except pd.errors.EmptyDataError:
            continue # Skip if a file is empty
        except Exception:
            # Silently skip any other error during processing of a single file pair
            continue

if __name__ == '__main__':
    main_processing_pipeline()
    import pandas as pd
    from pathlib import Path
    from tqdm import tqdm
    import datetime
    import os

def silent_process_flw_to_filtered_csv(flw_filepath_str, output_csv_filepath_str,
                                       filter_start_date_str, filter_end_date_str):
    """
    Extracts daily flow data from a .flw file, filters it by a date range,
    ensures all consecutive dates within the range are present, and saves it to a CSV file.
    This version is silent and does not print debug messages.
    """
    flw_filepath = Path(flw_filepath_str)
    output_csv_filepath = Path(output_csv_filepath_str)

    flw_start_date_obj = None
    current_date_obj = None
    data_lines_started = False
    raw_extracted_data = []

    try:
        filter_start_date = datetime.datetime.strptime(filter_start_date_str, "%Y-%m-%d").date()
        filter_end_date = datetime.datetime.strptime(filter_end_date_str, "%Y-%m-%d").date()
    except ValueError:
        # Invalid date format, cannot proceed for this file
        return

    try:
        with open(flw_filepath, 'r', encoding='utf-8') as f_in:
            for line_content in f_in:
                line = line_content.strip()
                if not line:
                    continue

                if line.startswith("FROM "):
                    parts = line.split()
                    if len(parts) >= 4:
                        try:
                            year, month, day = int(parts[1]), int(parts[2]), int(parts[3])
                            flw_start_date_obj = datetime.date(year, month, day)
                            current_date_obj = flw_start_date_obj
                        except ValueError:
                            return # Error parsing date
                    else:
                        return # Unexpected 'FROM' line format

                elif line.startswith("Flow(cms)"):
                    if flw_start_date_obj is None:
                        return # 'Flow(cms)' header found before 'FROM' date
                    data_lines_started = True
                    continue

                elif data_lines_started and current_date_obj:
                    parts = line.split()
                    if not parts:
                        continue
                    try:
                        flow_value = float(parts[0])
                        if filter_start_date <= current_date_obj <= filter_end_date:
                            raw_extracted_data.append((current_date_obj, flow_value))
                        
                        current_date_obj += datetime.timedelta(days=1)
                        
                        # Optimization: if current date is past filter end and last data point was for filter end
                        if current_date_obj > filter_end_date and raw_extracted_data and \
                           raw_extracted_data[-1][0] >= filter_end_date:
                            break
                    except (ValueError, IndexError):
                        # Silently ignore lines that cannot be parsed as flow data
                        pass
    except FileNotFoundError:
        return # Input file not found
    except Exception:
        return # Other unexpected error during file reading

    # Ensure output directory exists
    output_csv_filepath.parent.mkdir(parents=True, exist_ok=True)

    # Create a DataFrame with all dates in the filter range
    all_dates_pd_range = pd.date_range(start=filter_start_date_str, end=filter_end_date_str, freq='D')
    df_for_range = pd.DataFrame(index=all_dates_pd_range)
    df_for_range.index.name = 'Date'
    df_for_range['Flow'] = pd.NA # Initialize with NA

    if raw_extracted_data:
        df_extracted = pd.DataFrame(raw_extracted_data, columns=['Date', 'Flow'])
        df_extracted['Date'] = pd.to_datetime(df_extracted['Date'])
        df_extracted = df_extracted.set_index('Date')
        # Update the Flow column in df_for_range with extracted values
        df_for_range.update(df_extracted)

    df_output = df_for_range.reset_index()
    df_output['Date'] = df_output['Date'].dt.strftime('%Y-%m-%d')

    try:
        df_output.to_csv(output_csv_filepath, index=False, na_rep='NaN')
    except Exception:
        # Silently ignore errors during CSV writing for this file
        pass

def main_processing_pipeline():
    # --- Step 1: Convert .flw files to .csv in LBRM_discharge ---
    flow_dir = Path("../metadata/LBRM_Flow")
    discharge_dir = Path("../metadata/LBRM_discharge")
    discharge_dir.mkdir(parents=True, exist_ok=True)

    flw_files = list(flow_dir.glob("*.flw"))

    start_date_filter = "1980-01-01"
    end_date_filter = "2023-12-31"

    for flw_file_path in tqdm(flw_files, desc="Step 1: Converting FLW to CSV", unit="file"):
        basin_name = flw_file_path.stem
        output_csv_path = discharge_dir / f"{basin_name}.csv"
        silent_process_flw_to_filtered_csv(str(flw_file_path), str(output_csv_path),
                                           start_date_filter, end_date_filter)

    # --- Step 2: Merge converted .csv files with Daymet data ---
    daymet_dir = Path("../metadata/LBRM_daymet")

    converted_csv_files = list(discharge_dir.glob("*.csv"))

    for discharge_csv_path in tqdm(converted_csv_files, desc="Step 2: Merging CSV with Daymet", unit="file"):
        basin_name = discharge_csv_path.stem
        daymet_file_path = daymet_dir / f"{basin_name}.csv"

        if not daymet_file_path.exists():
            continue # Skip if corresponding Daymet file doesn't exist

        try:
            discharge_df = pd.read_csv(discharge_csv_path)
            if 'Date' not in discharge_df.columns or 'Flow' not in discharge_df.columns:
                continue # Skip if required columns are missing in discharge file

            daymet_df = pd.read_csv(daymet_file_path)
            if 'time' not in daymet_df.columns:
                continue # Skip if 'time' column is missing in Daymet file

            # Prepare for merge
            # 'Date' in discharge_df is already YYYY-MM-DD string from conversion step
            discharge_df['merge_date_key'] = discharge_df['Date']
            daymet_df['merge_date_key'] = pd.to_datetime(daymet_df['time']).dt.strftime('%Y-%m-%d')

            # Merge. 'Date' column from discharge_df will be used.
            merged_df = pd.merge(
                daymet_df,
                discharge_df[['merge_date_key', 'Date', 'Flow']],
                on='merge_date_key',
                how='left'
            )

            # Post-merge processing
            merged_df.rename(columns={'Flow': 'discharge'}, inplace=True)

            daymet_original_cols = [col for col in daymet_df.columns if col not in ['time', 'merge_date_key']]
            
            # Define final column order, ensuring 'Date' (from discharge_df) is first
            final_columns = ['Date'] + daymet_original_cols + ['discharge']
            
            # Ensure all specified final columns exist in merged_df, adding them with NA if missing
            # This is a safeguard, typically merge should handle this.
            for col in final_columns:
                if col not in merged_df.columns:
                    merged_df[col] = pd.NA
            
            output_df = merged_df[final_columns]

            # Save, overwriting the Daymet file
            output_df.to_csv(daymet_file_path, index=False, na_rep='NaN')

        except pd.errors.EmptyDataError:
            continue # Skip if a file is empty
        except Exception:
            # Silently skip any other error during processing of a single file pair
            continue

if __name__ == '__main__':
    main_processing_pipeline()

Step 2: Merging CSV with Daymet: 100%|██████████| 110/110 [00:02<00:00, 37.57file/s]


In [21]:
attributes = pd.read_csv("../metadata/attributes_LBRM.csv")

In [23]:
import pandas as pd
import re

# Define the filepath
filepath = "../metadata/attributes_LBRM.csv"

try:
    # Load the dataframe
    attributes_df = pd.read_csv(filepath)
except FileNotFoundError:
    print(f"Error: File not found at {filepath}")
    exit()
except Exception as e:
    print(f"Error loading CSV: {e}")
    exit()

# --- 1. Revise the content of `gauge_id` column ---
prefix_map = {
    "ER": "eri",
    "GEO": "geo",
    "HU": "hur",
    "MIC": "mic",
    "ON": "on",
    "STC": "stc",
    "SUP": "sup"
}

def transform_gauge_id(gid):
    if pd.isna(gid):
        return gid # Keep NaN as is

    # Remove "subdata_" prefix
    if isinstance(gid, str) and gid.startswith("subdata_"):
        gid = gid.replace("subdata_", "", 1)
    else:
        # If it doesn't start with "subdata_", or not a string,
        # try to process as is or return original if format is unexpected
        pass

    # Use regex to find the prefix and number part
    # This regex looks for 2 or 3 uppercase letters followed by one or more digits
    match = re.match(r"([A-Z]{2,3})(\d+)", str(gid)) # Ensure gid is string for regex

    if match:
        original_prefix = match.group(1)
        number_str = match.group(2)

        if original_prefix in prefix_map:
            new_prefix = prefix_map[original_prefix]

            # Format number: add leading zero if single digit and not already zero
            if len(number_str) == 1 and number_str != '0':
                new_number_str = "0" + number_str
            else:
                new_number_str = number_str
            
            return new_prefix + new_number_str
        else:
            # If prefix not in map, return the id after "subdata_" removal
            return gid 
    else:
        # If pattern doesn't match (e.g., already processed or different format)
        return gid # Return original or "subdata_" stripped version

# Apply the transformation
if 'gauge_id' in attributes_df.columns:
    attributes_df['gauge_id'] = attributes_df['gauge_id'].apply(transform_gauge_id)
else:
    print("Warning: 'gauge_id' column not found in the DataFrame.")


# --- 2. Normalize all the columns except `gauge_id` column ---
columns_to_normalize = [col for col in attributes_df.columns if col != 'gauge_id']

for col in columns_to_normalize:
    if pd.api.types.is_numeric_dtype(attributes_df[col]):
        mean_val = attributes_df[col].mean()
        attributes_df[col] = attributes_df[col] - mean_val
    else:
        print(f"Warning: Column '{col}' is not numeric and will not be normalized.")

# --- 3. Replace the original csv file with the updated one, discarding the index ---
try:
    attributes_df.to_csv(filepath, index=False)
    # print(f"Successfully processed and saved the updated data to {filepath}") # Optional: for confirmation
except Exception as e:
    print(f"Error saving CSV: {e}")


In [36]:
import xarray as xr
import os
import pandas as pd
from pathlib import Path
from tqdm import tqdm
import numpy as np  # For log and numerical operations

# Epsilon for log transformation to avoid log(0)
LOG_EPSILON = 1e-6

def DataFrame_to_CDF(data: pd.DataFrame, output_dir: str, output_name: str):
    # Ensure index is named "date" (not "Date")
    if data.index.name != 'date':
        raise ValueError("Index must be named 'date'")
    # Ensure index is datetime
    data.index = pd.to_datetime(data.index, errors='coerce')
    # Infer frequency or assume daily
    freq = pd.infer_freq(data.index)
    if freq is None:
        data.index = pd.date_range(start=data.index.min(), periods=len(data.index), freq='D')
    os.makedirs(output_dir, exist_ok=True)
    # Use to_xarray() instead:
    ds = data.to_xarray()  # This should preserve the index name "date"
    ds.to_netcdf(os.path.join(output_dir, f"{output_name}.nc"))

def process_and_normalize_data():
    csv_input_dir = Path("../metadata/LBRM_daymet")
    netcdf_output_dir = Path("../data/time_series")
    stats_file_path = Path("LBRM_stats.txt")
    log_file_path = Path("test.txt")

    if not csv_input_dir.exists():
        return

    netcdf_output_dir.mkdir(parents=True, exist_ok=True)
    
    csv_files = list(csv_input_dir.glob("*.csv"))
    if not csv_files:
        with open(log_file_path, 'w') as f:
            pass
        with open(stats_file_path, 'w') as f:
            f.write("variable,mean,variance\n")
        return

    log_transform_cols = ['discharge', 'prcp', 'swe']
    
    # --- Pass 1: Calculate global statistics ---
    col_sums = {}
    col_sum_sqs = {}
    col_counts = {}
    # This set will store the names as they appear in stats file (e.g. prcp_log or tmax)
    stat_col_names_set = set()

    for csv_file_path in tqdm(csv_files, desc="Pass 1: Calculating Stats", unit="file"):
        try:
            df = pd.read_csv(csv_file_path)
            # Rename "Date" to "date" so that further processing is consistent.
            if 'Date' in df.columns:
                df.rename(columns={'Date': 'date'}, inplace=True)
            elif 'date' not in df.columns:
                continue

            temp_stat_df = pd.DataFrame()  # Create a temporary df for stat calculation columns

            # Handle log transformations for stat calculation
            for col in log_transform_cols:
                if col in df.columns and pd.api.types.is_numeric_dtype(df[col]):
                    stat_col_name = col + '_log'
                    temp_stat_df[stat_col_name] = np.log(df[col] + LOG_EPSILON)
                    stat_col_names_set.add(stat_col_name)
            
            # Handle other numeric columns for stat calculation
            for col in df.columns:
                if col not in ['date'] and col not in log_transform_cols and pd.api.types.is_numeric_dtype(df[col]):
                    temp_stat_df[col] = df[col]
                    stat_col_names_set.add(col)

            # Accumulate stats from temp_stat_df
            for col_name in temp_stat_df.columns:
                valid_data = temp_stat_df[col_name].dropna()
                if not valid_data.empty:
                    col_sums[col_name] = col_sums.get(col_name, 0) + valid_data.sum()
                    col_sum_sqs[col_name] = col_sum_sqs.get(col_name, 0) + (valid_data**2).sum()
                    col_counts[col_name] = col_counts.get(col_name, 0) + valid_data.count()
        except Exception:
            continue
            
    global_means = {}
    global_vars = {}

    for col_name in stat_col_names_set:
        if col_name in col_counts and col_counts[col_name] > 0:
            mean = col_sums[col_name] / col_counts[col_name]
            variance = (col_sum_sqs[col_name] / col_counts[col_name]) - (mean**2)
            variance = max(0, variance) 
            global_means[col_name] = mean
            global_vars[col_name] = variance
        else:
            global_means[col_name] = 0 
            global_vars[col_name] = 0

    with open(stats_file_path, 'w') as f_stats:
        f_stats.write("variable,mean,variance\n")
        for col_name in sorted(list(stat_col_names_set)):
            f_stats.write(f"{col_name},{global_means.get(col_name,0)},{global_vars.get(col_name,0)}\n")

    # --- Pass 2: Normalize data and save to NetCDF ---
    processed_filenames = []
    for csv_file_path in tqdm(csv_files, desc="Pass 2: Normalizing & Saving", unit="file"):
        base_filename = csv_file_path.stem
        try:
            df = pd.read_csv(csv_file_path)
            # Rename "Date" to "date" for consistency
            if 'Date' in df.columns:
                df.rename(columns={'Date': 'date'}, inplace=True)
            elif 'date' not in df.columns:
                continue
            
            df['date'] = pd.to_datetime(df['date'], errors='coerce')
            df.dropna(subset=['date'], inplace=True)
            if df.empty:
                continue
            
            # Create a DataFrame for processing, will be modified in place
            processed_df = df.copy()
            
            # Columns that will be in the NetCDF output (original names)
            final_numeric_cols_for_netcdf = []

            # Normalize log-transformed columns
            for original_col_name in log_transform_cols:
                if original_col_name in processed_df.columns and pd.api.types.is_numeric_dtype(processed_df[original_col_name]):
                    stat_col_name = original_col_name + '_log'  # Name used in stats
                    
                    if stat_col_name in global_means and stat_col_name in global_vars:
                        log_transformed_data = np.log(processed_df[original_col_name] + LOG_EPSILON)
                        
                        mean = global_means[stat_col_name]
                        variance = global_vars[stat_col_name]
                        std_dev = np.sqrt(variance)
                        
                        if std_dev > 0:
                            processed_df[original_col_name] = (log_transformed_data - mean) / std_dev
                        else:
                            processed_df[original_col_name] = 0.0
                        final_numeric_cols_for_netcdf.append(original_col_name)

            # Normalize other numeric columns (not log-transformed)
            for original_col_name in processed_df.columns:
                if original_col_name not in ['date'] and original_col_name not in log_transform_cols and \
                   pd.api.types.is_numeric_dtype(processed_df[original_col_name]):
                    
                    if original_col_name in global_means and original_col_name in global_vars:  # Stats are for original name
                        mean = global_means[original_col_name]
                        variance = global_vars[original_col_name]
                        std_dev = np.sqrt(variance)

                        if std_dev > 0:
                            processed_df[original_col_name] = (processed_df[original_col_name] - mean) / std_dev
                        else:
                            processed_df[original_col_name] = 0.0
                        final_numeric_cols_for_netcdf.append(original_col_name)

            # Prepare for DataFrame_to_CDF
            processed_df.set_index('date', inplace=True)
            processed_df.index.name = 'date'
            
            # Select only the processed numeric columns for the NetCDF file
            final_df_for_nc = processed_df[list(dict.fromkeys(final_numeric_cols_for_netcdf))]

            if not final_df_for_nc.empty and final_df_for_nc.index.name == 'date':
                DataFrame_to_CDF(final_df_for_nc, str(netcdf_output_dir), base_filename)
                processed_filenames.append(base_filename)

        except Exception:
            continue

    with open(log_file_path, 'w') as f_log:
        for name in processed_filenames:
            f_log.write(name + "\n")

if __name__ == "__main__":
    process_and_normalize_data()

Pass 1: Calculating Stats:   0%|          | 0/121 [00:00<?, ?file/s]C:\Users\ybrot\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\pandas\core\arraylike.py:399: RuntimeWarning: invalid value encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)
Pass 1: Calculating Stats:   4%|▍         | 5/121 [00:00<00:02, 49.61file/s]C:\Users\ybrot\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\pandas\core\arraylike.py:399: RuntimeWarning: invalid value encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)
Pass 1: Calculating Stats:   8%|▊         | 10/121 [00:00<00:02, 48.29file/s]C:\Users\ybrot\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\pandas\core\arraylike.py:399: RuntimeWarning: invalid value encountered in log
  result = getattr(

In [30]:
def read_CDF(file_path: str) -> pd.DataFrame:
    """
    Reads a NetCDF file and converts it back to a pandas DataFrame.
    
    Parameters
    ----------
    file_path : str
        The path to the NetCDF file.
    
    Returns
    ----------
    pd.DataFrame
        A DataFrame containing the data from the NetCDF file.
    """
    ds = xr.open_dataset(file_path)
    return ds.to_dataframe()

In [2]:
import xarray as xr
import os

nc_file = "../data/time_series/eri01.nc"

# Ensure the dataset is properly closed after processing using a context manager.
with xr.open_dataset(nc_file) as ds:
    # Rename coordinate if needed
    if "index" in ds.coords:
        ds = ds.rename({"index": "date"})
    if "index" in ds.dims:
        ds = ds.rename_dims({"index": "date"})
    # Save to a temporary file
    temp_file = nc_file + ".tmp"
    ds.to_netcdf(temp_file)

# Replace the original file
os.replace(temp_file, nc_file)

PermissionError: [WinError 5] 액세스가 거부되었습니다: '../data/time_series/eri01.nc.tmp' -> '../data/time_series/eri01.nc'

In [42]:
df = read_CDF("../data/time_series/eri01.nc")
df

,discharge,prcp,swe,dayl,srad,tmax,tmin,vp
index,,,,,,,,
1980-01-01,-0.700314,-1.064147,0.963311,-1.292458,-1.501569,-1.116206,-0.544622,-0.717958
1980-01-02,-0.760434,0.738616,0.945896,-1.287093,-1.464363,-1.024746,-0.461497,-0.663335
1980-01-03,-0.813997,-1.064147,0.939430,-1.281291,-1.284328,-1.097606,-0.619547,-0.764652
1980-01-04,-0.892854,0.784975,0.952303,-1.275054,-1.461757,-1.312947,-0.787144,-0.860849
1980-01-05,-0.974515,0.850082,0.971022,-1.268388,-0.793207,-1.149317,-0.922001,-0.929904
...,...,...,...,...,...,...,...,...
2023-12-16,-0.110805,0.871575,-0.910645,-1.312592,-1.724612,-0.234643,0.478629,0.244297
2023-12-17,-0.063403,1.110282,-0.910645,-1.309474,-1.724334,-0.337953,0.299351,0.022792
2023-12-18,0.209335,0.885168,-0.910645,-1.305901,-1.709531,-0.568704,0.126307,-0.166463
